# 04_silver_to_gold
Third task of the job `gharchive_pipeline` (runs after `bronze_to_silver`).
It rebuilds every Gold table from Silver with a **full refresh** (`INSERT OVERWRITE`), so every new
incremental load automatically refreshes the dashboard data, and a rerun always gives the same result.

Order: `dim_date`, `dim_hour`, `dim_event_type`, `dim_repo`, `fact_activity_hourly`, `fact_repo_daily`,
`fact_repo_trend`, `fact_daily_summary`.

Logging: every table writes one row to `pipeline_execution_logs` (layer `Silver-to-Gold`, rows written,
start and end time, Success or Failure) and one row to `gold_aggregation_metrics` (Silver rows aggregated, seconds).
If one table fails, the others still run, and at the end the notebook raises so the job turns red.

In [0]:
%run ./common

In [0]:
%run ./common_gold

In [0]:
# Parameters. The job passes the same job parameters to every task.
# Gold is always a full refresh, so start_hour and end_hour are not used here. mode is only written to the log.
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_hour", "")
dbutils.widgets.text("end_hour", "")
dbutils.widgets.dropdown("force_reprocess", "false", ["false", "true"])

mode = dbutils.widgets.get("mode").strip().lower()
load_type = "full-refresh (" + mode + " run)"
run_id = str(uuid.uuid4())
print("run_id =", run_id, "| load_type =", load_type)

## Source: Silver events
Only the columns Gold needs are selected (column pruning: Delta reads fewer bytes).
`date_key` (yyyymmdd) and `hour_key` (0 to 23, UTC) are the keys of the date and hour dimensions.

In [0]:
events = spark.table(SILVER_EVENT).select(
    "event_id", "event_type", "actor_id", "repo_id", "repo_name", "created_at", "event_date"
)
events = events.filter(F.col("event_date").isNotNull() & F.col("repo_id").isNotNull())
events = events.withColumn("date_key", F.date_format(F.col("event_date"), "yyyyMMdd").cast("int"))
events = events.withColumn("hour_key", F.hour(F.col("created_at")))

bounds = events.agg(
    F.min("event_date").alias("date_from"),
    F.max("event_date").alias("date_to"),
    F.count(F.lit(1)).alias("row_count"),
).collect()[0]

date_from = bounds["date_from"]
date_to = bounds["date_to"]
source_rows = bounds["row_count"]
if source_rows == 0:
    raise Exception("silver_event is empty. Run notebooks 01 and 02 first.")

date_range_text = str(date_from) + " to " + str(date_to)
# Day number since 1970-01-01. Used by the trend windows and to know when 7 earlier days exist.
first_day_num = (date_from - date(1970, 1, 1)).days

print("Silver rows in scope:", source_rows)
print("Date range:", date_range_text)

## Step runner
Builds one table, overwrites it, writes the two log rows. A failure is logged and the next table still runs.

In [0]:
failed_tables = []


def gold_step(table_name, build_function, view_name):
    start_time = datetime.now()
    try:
        df = build_function()
        rows_written = write_gold_table(df, table_name, view_name)
        write_log(run_id, LAYER_GOLD, table_name, date_range_text, load_type, start_time,
                  "Success", rows_written, 0, 0)
        write_gold_metrics(run_id, table_name, date_from, date_to, source_rows, rows_written, start_time)
        seconds = (datetime.now() - start_time).total_seconds()
        print("OK  ", table_name, "| rows =", rows_written, "| seconds =", round(seconds, 1))
    except Exception as e:
        write_log(run_id, LAYER_GOLD, table_name, date_range_text, load_type, start_time,
                  "Failure", 0, 0, 0, str(e)[:2000])
        failed_tables.append(table_name)
        print("FAIL", table_name, str(e)[:300])

## dim_date
One row per calendar day between the first and last event date (no gaps, so charts show quiet days too).

In [0]:
def build_dim_date():
    days = spark.sql(
        "SELECT explode(sequence(DATE'" + str(date_from) + "', DATE'" + str(date_to) + "', INTERVAL 1 DAY)) AS full_date"
    )
    return days.select(
        F.date_format(F.col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
        F.col("full_date"),
        F.year(F.col("full_date")).alias("year"),
        F.month(F.col("full_date")).alias("month_num"),
        F.date_format(F.col("full_date"), "MMMM").alias("month_name"),
        F.dayofmonth(F.col("full_date")).alias("day_of_month"),
        F.dayofweek(F.col("full_date")).alias("day_of_week_num"),
        F.date_format(F.col("full_date"), "EEEE").alias("day_name"),
        F.weekofyear(F.col("full_date")).alias("week_of_year"),
        F.dayofweek(F.col("full_date")).isin(1, 7).alias("is_weekend"),
        F.current_timestamp().alias("load_timestamp"),
    )


gold_step(DIM_DATE, build_dim_date, "gold_dim_date")

## dim_hour
24 rows. GH Archive hours are UTC; `hour_pkt` gives the Pakistan time hour for the heat map.

In [0]:
def build_dim_hour():
    hours = spark.range(24).select(F.col("id").cast("int").alias("hour_key"))
    hours = hours.withColumn("hour_label", F.concat(F.lpad(F.col("hour_key").cast("string"), 2, "0"), F.lit(":00 UTC")))
    hours = hours.withColumn("hour_pkt", ((F.col("hour_key") + 5) % 24).cast("int"))
    hours = hours.withColumn(
        "day_part_pkt",
        F.when(F.col("hour_pkt") < 6, F.lit("Night"))
        .when(F.col("hour_pkt") < 12, F.lit("Morning"))
        .when(F.col("hour_pkt") < 18, F.lit("Afternoon"))
        .otherwise(F.lit("Evening"))
    )
    hours = hours.withColumn("load_timestamp", F.current_timestamp())
    return hours


gold_step(DIM_HOUR, build_dim_hour, "gold_dim_hour")

## dim_event_type
Every event type seen in Silver, with a business category.

In [0]:
def build_dim_event_type():
    types = events.select("event_type").distinct()
    types = types.withColumn("event_category", event_category_column())
    types = types.withColumn("load_timestamp", F.current_timestamp())
    return types


gold_step(DIM_EVENT_TYPE, build_dim_event_type, "gold_dim_event_type")

## dim_repo (tracked repositories only)
One row per repository that passes the "tracked" rule in `common_gold`.
`max_by(repo_name, created_at)` keeps the latest name, because a repository can be renamed.

In [0]:
def build_dim_repo():
    is_community = F.col("event_type").isin(COMMUNITY_EVENT_TYPES)
    stats = events.groupBy("repo_id").agg(
        F.expr("max_by(repo_name, created_at)").alias("repo_name"),
        F.min("event_date").alias("first_seen_date"),
        F.max("event_date").alias("last_seen_date"),
        F.count(F.lit(1)).alias("total_events"),
        F.countDistinct("actor_id").alias("distinct_actors"),
        F.sum(F.when(is_community, 1).otherwise(0)).cast("bigint").alias("community_events"),
    )
    tracked = stats.filter(
        (F.col("distinct_actors") >= TRACKED_MIN_ACTORS)
        | (F.col("community_events") >= TRACKED_MIN_COMMUNITY_EVENTS)
    )
    tracked = tracked.withColumn("owner", F.split(F.col("repo_name"), "/").getItem(0))
    tracked = tracked.withColumn("repo_short_name", F.split(F.col("repo_name"), "/").getItem(1))
    tracked = tracked.withColumn("load_timestamp", F.current_timestamp())
    return tracked


gold_step(DIM_REPO, build_dim_repo, "gold_dim_repo")

## fact_activity_hourly
Grain: one row per day x hour x event type, over ALL repositories. Powers the overview and the heat map.

In [0]:
def build_fact_activity_hourly():
    hourly = events.groupBy("date_key", "hour_key", "event_type").agg(
        F.count(F.lit(1)).alias("events"),
        F.countDistinct("repo_id").alias("distinct_repos"),
        F.countDistinct("actor_id").alias("distinct_actors"),
    )
    hourly = hourly.withColumn("load_timestamp", F.current_timestamp())
    return hourly


gold_step(FACT_ACTIVITY_HOURLY, build_fact_activity_hourly, "gold_fact_activity_hourly")

## fact_repo_daily
Grain: one row per tracked repository per day (only days with at least one event).
Event counts by type come from `silver_event`. Opened and closed counts come from the actions in
`silver_pull_request` and `silver_issue`, joined on `event_id`.

In [0]:
def build_fact_repo_daily():
    tracked_ids = spark.table(DIM_REPO).select("repo_id")
    tracked_events = events.join(tracked_ids, "repo_id", "left_semi")

    counts = tracked_events.groupBy("date_key", "repo_id").agg(
        F.count(F.lit(1)).alias("total_events"),
        count_type(["PushEvent"], "push_events"),
        count_type(["PullRequestEvent"], "pull_request_events"),
        count_type(["PullRequestReviewEvent", "PullRequestReviewCommentEvent"], "pr_review_events"),
        count_type(["IssuesEvent"], "issue_events"),
        count_type(["IssueCommentEvent"], "issue_comment_events"),
        count_type(["ReleaseEvent"], "release_events"),
        count_type(["WatchEvent"], "watch_events"),
        count_type(["ForkEvent"], "fork_events"),
        count_type(["CreateEvent"], "create_events"),
        count_type(["DeleteEvent"], "delete_events"),
        F.countDistinct("actor_id").alias("distinct_actors"),
    )

    # Pull requests opened and closed
    pr_events = tracked_events.filter(F.col("event_type") == "PullRequestEvent").select("event_id", "date_key", "repo_id")
    pr_actions = spark.table(SILVER_PR).select("event_id", "action").join(pr_events, "event_id")
    pr_counts = pr_actions.groupBy("date_key", "repo_id").agg(
        count_value("action", "opened", "prs_opened"),
        count_value("action", "closed", "prs_closed"),
    )

    # Issues opened and closed
    issue_events = tracked_events.filter(F.col("event_type") == "IssuesEvent").select("event_id", "date_key", "repo_id")
    issue_actions = spark.table(SILVER_ISSUE).select("event_id", "action").join(issue_events, "event_id")
    issue_counts = issue_actions.groupBy("date_key", "repo_id").agg(
        count_value("action", "opened", "issues_opened"),
        count_value("action", "closed", "issues_closed"),
    )

    daily = counts.join(pr_counts, ["date_key", "repo_id"], "left")
    daily = daily.join(issue_counts, ["date_key", "repo_id"], "left")
    daily = daily.fillna(0, subset=["prs_opened", "prs_closed", "issues_opened", "issues_closed"])
    daily = daily.withColumn("load_timestamp", F.current_timestamp())
    return daily


gold_step(FACT_REPO_DAILY, build_fact_repo_daily, "gold_fact_repo_daily")

## fact_repo_trend (surges and momentum)
Grain: same as `fact_repo_daily` (tracked repo x active day). Built from `fact_repo_daily` with window functions
over each repository's own history, ordered by day number:
* baseline = the previous 7 calendar days (`rangeBetween(-7, -1)`). Days without events are missing rows, so the
  average and standard deviation are calculated from the sum and the sum of squares divided by 7, which counts
  quiet days as 0.
* momentum = events in the last 7 days (`rangeBetween(-6, 0)`) against the 7 days before (`rangeBetween(-13, -7)`).
* Days in the first week of data have no full baseline, so they never count as surges.

In [0]:
def build_fact_repo_trend():
    daily = spark.table(FACT_REPO_DAILY).select("date_key", "repo_id", F.col("total_events").alias("events"))
    daily = daily.withColumn("full_date", F.to_date(F.col("date_key").cast("string"), "yyyyMMdd"))
    daily = daily.withColumn("day_num", F.datediff(F.col("full_date"), F.to_date(F.lit("1970-01-01"))))
    daily = daily.withColumn("events_d", F.col("events").cast("double"))

    by_repo = Window.partitionBy("repo_id").orderBy("day_num")
    w_baseline = by_repo.rangeBetween(-7, -1)
    w_last_7 = by_repo.rangeBetween(-6, 0)
    w_prev_7 = by_repo.rangeBetween(-13, -7)

    t = daily.withColumn("base_sum", F.coalesce(F.sum("events_d").over(w_baseline), F.lit(0.0)))
    t = t.withColumn("base_sumsq", F.coalesce(F.sum(F.col("events_d") * F.col("events_d")).over(w_baseline), F.lit(0.0)))
    t = t.withColumn("events_last_7d", F.sum("events").over(w_last_7))
    t = t.withColumn("events_prev_7d", F.coalesce(F.sum("events").over(w_prev_7), F.lit(0)).cast("bigint"))

    # Average and sample standard deviation over 7 calendar days
    t = t.withColumn("baseline_avg", F.col("base_sum") / F.lit(7.0))
    variance = (F.col("base_sumsq") - F.lit(7.0) * F.col("baseline_avg") * F.col("baseline_avg")) / F.lit(6.0)
    t = t.withColumn("baseline_std", F.sqrt(F.greatest(variance, F.lit(0.0))))

    t = t.withColumn("has_full_baseline", F.col("day_num") >= F.lit(first_day_num + 7))
    has_full_momentum = F.col("day_num") >= F.lit(first_day_num + 13)

    t = t.withColumn(
        "z_score",
        F.when(F.col("has_full_baseline") & (F.col("baseline_std") > 0),
               (F.col("events_d") - F.col("baseline_avg")) / F.col("baseline_std"))
    )
    t = t.withColumn(
        "pct_vs_baseline",
        F.when(F.col("has_full_baseline") & (F.col("baseline_avg") > 0),
               (F.col("events_d") - F.col("baseline_avg")) / F.col("baseline_avg") * 100.0)
    )
    t = t.withColumn(
        "growth_pct_7d",
        F.when(has_full_momentum & (F.col("events_prev_7d") > 0),
               (F.col("events_last_7d") - F.col("events_prev_7d")) / F.col("events_prev_7d") * 100.0)
    )

    big_enough = F.col("events") >= F.lit(SURGE_MIN_EVENTS)
    z_rule = (F.col("baseline_std") > 0) & (F.col("z_score") >= F.lit(SURGE_MIN_Z))
    flat_rule = (F.col("baseline_std") == 0) & (
        F.col("events_d") >= F.lit(SURGE_FLAT_MULTIPLIER) * F.greatest(F.col("baseline_avg"), F.lit(1.0))
    )
    surge = F.col("has_full_baseline") & big_enough & (z_rule | flat_rule)
    t = t.withColumn("is_surge", F.coalesce(surge, F.lit(False)))

    t = t.withColumn("load_timestamp", F.current_timestamp())
    return t


gold_step(FACT_REPO_TREND, build_fact_repo_trend, "gold_fact_repo_trend")

## fact_daily_summary
Grain: one row per day. Daily distinct counts (repos, contributors) cannot be added up from hourly rows,
so they are computed here once per day for the KPI cards.

In [0]:
def build_fact_daily_summary():
    daily = events.groupBy("date_key").agg(
        F.count(F.lit(1)).alias("events"),
        F.countDistinct("repo_id").alias("active_repos"),
        F.countDistinct("actor_id").alias("active_actors"),
    )
    tracked_daily = spark.table(FACT_REPO_DAILY).groupBy("date_key").agg(
        F.count(F.lit(1)).alias("active_tracked_repos")
    )
    summary = daily.join(tracked_daily, "date_key", "left")
    summary = summary.fillna(0, subset=["active_tracked_repos"])
    summary = summary.withColumn("load_timestamp", F.current_timestamp())
    return summary


gold_step(FACT_DAILY_SUMMARY, build_fact_daily_summary, "gold_fact_daily_summary")

## Result

In [0]:
if len(failed_tables) > 0:
    raise Exception("Silver-to-Gold failed for: " + ", ".join(failed_tables))

display(spark.sql(
    "SELECT layer, target_table, parameter_processed, load_type, status, rows_inserted, start_time, end_time "
    "FROM " + EXEC_LOG + " WHERE run_id = '" + run_id + "' ORDER BY start_time"
))